# Proyecto Final: Automatización de Análisis y Creación de Contenido para Streaming

**Alumno:** Joaquín Spero | **Curso:** Inteligencia artificial: Generación de Prompts | **Comisión:** 95920

## 1. Resumen e Introducción
**Presentación del problema:** Como creador de contenido, invierto horas analizando qué momentos del stream funcionaron mejor basándome en el chat y las métricas de retención, para luego adaptar ese contenido a YouTube. Es un proceso manual, repetitivo y lento.

**Propuesta de solución:** Utilizo la API de Gemini y la Librería Pandas IA para procesar datos tabulares y texto no estructurado. Además, integro un modelo texto a imagen para la generación rápida de miniaturas.

**Viabilidad:** El proyecto es técnica y económicamente viable. Utilizo Python, la API gratuita de Gemini (gemini-3.8-flash) y PandasAI ejecutados localmente. El costo es cero.

## 2. Objetivos
* Reducir el tiempo de post-producción y análisis de mis streams.
* Automatizar el análisis de los momentos clave extrayendo insights del chat.
* Obtener datos sobre la audiencia mediante consultas en lenguaje natural.
* Generar assets gráficos para las miniaturas mediante prompts optimizados.

## 3. Metodología y Evaluación de Costos
Llevo a cabo el proyecto mediante esta Prueba de Concepto (POC). El código está optimizado para ejecutar solo 2 consultas a la API en toda la ejecución (una de texto a texto y otra interna de PandasAI), garantizando consumo nulo de saldo y máxima rentabilidad.

## 4. Herramientas y tecnologías (Fast Prompting)
* **Few shoot prompting:** Para estructurar la salida del resumen del chat sin alucinaciones.
* **Zero prompt shooting (Librería Pandas IA):** Para consultar el CSV sin ejemplos previos.
* **Prompt iterativo y dirigido:** Para la generación de la miniatura en Bing Image Creator.

---
### Fase 1: Configuración e inicialización

In [ ]:
import pandas as pd
from google import genai
from pandasai import SmartDataframe
from pandasai.llm import GoogleGemini

api_key = "TU_API_KEY_GEMINI"
client = genai.Client(api_key=api_key)

### Fase 2: Análisis de Sentimiento y Títulos (Few shoot prompting)
Procesamiento del log de Twitch utilizando el modelo texto a texto.

In [ ]:
with open('chat_log.txt', 'r', encoding='utf-8') as file:
    chat_content = file.read()

prompt = f"""
Sos un asistente experto en marketing de YouTube. Lee el siguiente log de chat de Twitch y generá 1 título clickbait y un breve resumen.
Ejemplo:
Chat: "[10:00] user: me muero, [10:01] user2: épico"
Salida: Título: ¡EL MEJOR MOMENTO DEL STREAM! - Resumen: La audiencia reaccionó con mucha emoción a una jugada.

Chat a analizar:
{chat_content}
"""

response = client.models.generate_content(
    model='gemini-3.8-flash',
    contents=prompt
)
print(response.text)

### Fase 3: Análisis de Métricas (Librería Pandas IA + Zero prompt shooting)
Consultas en lenguaje natural al archivo CSV de métricas del stream.

In [12]:
df = pd.read_csv('stream_metrics.csv')

class GoogleGeminiFix(GoogleGemini):
    """Parche: fuerza el uso del SDK nuevo (google-genai) y respeta el modelo pedido."""
    def __init__(self, api_key: str, model: str, **kwargs):
        super().__init__(api_key=api_key, **kwargs)
        self.model = model
        self._client = genai.Client(api_key=api_key)

    def _generate_text(self, prompt: str, *args, **kwargs) -> str:
        response = self._client.models.generate_content(
            model=self.model,
            contents=prompt,
        )
        return response.text

llm = GoogleGeminiFix(api_key=api_key, model="gemini-3.8-flash")
df_smart = SmartDataframe(df, config={"llm": llm})

respuesta = df_smart.chat("¿Cuál fue el día con el pico máximo de espectadores y cuántos ingresos generó?")
print(respuesta)

El día con el pico máximo de espectadores (315 espectadores) fue el 2026-09-24 y generó 40.5 USD en ingresos.


### Generación de Miniatura (Modelo Texto a Imagen)
**Herramienta utilizada:** Bing Image Creator
**Técnica:** Prompt Iterativo y Dirigido

**Prompt final que utilicé:**
> "Una miniatura para YouTube de un stream de gaming, colores neón violeta y azul. En el centro un trofeo brillante con el logo de Twitch. Estilo render 3D de alta calidad, iluminación dramática."

**Resultado:**
![Miniatura generada](miniatura.png)